In [1]:
import sys
from pathlib import Path
import torch
from transformers import TrainingArguments

# Aggiungi src al path
ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from teacher_finetune import (
    load_chunks_dataset,
    TeacherModelConfig,
    build_teacher_tokenizer,
    build_teacher_model,
    freeze_bert_layers,
    build_collator,
    WeightedBCETrainer,
    compute_metrics,
    bf16_supported
)

paths = get_paths(ROOT)
print(f"Working on: {paths.root}")
print(f"GPU Available: {torch.cuda.is_available()}")

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working on: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526
GPU Available: True


In [3]:
# Definizione percorsi
train_chunks = paths.data_processed / "train_80k_chunks.parquet" # Assumo tu stia usando il 120k
train_counts = paths.data_processed / "train_120k_chunk_counts.parquet"
val_chunks   = paths.data_processed / "val_80k_chunks.parquet" # O il corrispettivo validation

# Caricamento Dataset (veloce perché memory-mapped)
train_ds = load_chunks_dataset(
    chunks_parquet=train_chunks,
    chunk_counts_parquet=train_counts,
    add_sample_weight=True, # IMPORTANTE: Bilancia i chunk
)

val_ds = load_chunks_dataset(
    chunks_parquet=val_chunks,
    add_sample_weight=False, # Validation standard
)

print(f"Train size: {len(train_ds)}")
print(f"Val size: {len(val_ds)}")

Cast labels: 100%|██████████| 26663/26663 [00:00<00:00, 39642.19 examples/s]

Train size: 212943
Val size: 26663


In [ ]:
# Cella 3: Costruzione Modello & FREEZING
MODEL_NAME = "bert-large-uncased"

tokenizer = build_teacher_tokenizer(MODEL_NAME)
collator = build_collator(tokenizer)

# Configura il modello
model_cfg = TeacherModelConfig(
    model_name=MODEL_NAME,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    gradient_checkpointing=True      # TRUE per salvare VRAM
)

model = build_teacher_model(model_cfg)

# ==========================================
# ❄️ FREEZING STRATEGICO ❄️
# ==========================================
freeze_bert_layers(model, freeze_embeddings=True, freeze_layers=20)

# ==========================================
# 🛠️ FIX PER IL WARNING "None of the inputs..."
# ==========================================
# Questo è obbligatorio quando usi Gradient Checkpointing + Freezing
if model_cfg.gradient_checkpointing:
    model.gradient_checkpointing_enable() 
    model.enable_input_require_grads()  # <--- QUESTA è la riga magica che risolve il problema
    print("✅ Gradient Checkpointing abilitato correttamente (con input_require_grads).")

# Ignora il warning sul tokenizer deprecated, è solo un avviso per il futuro (versione 5.0)
# Se vuoi pulirlo, nella cella successiva cambia 'tokenizer=tokenizer' in 'processing_class=tokenizer' 
# dentro WeightedBCETrainer, ma non è urgente.

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Embeddings FROZEN.
✅ First 20/24 encoder layers FROZEN.
📊 Params Status: 51.4M Trainable / 335.1M Total (15.3%)


In [6]:
# Percorso output
OUT_DIR = paths.checkpoints / "teacher_bert_large_frozen"

# Configurazione Iperparametri Anti-Overfitting
args = TrainingArguments(
    output_dir=str(OUT_DIR),
    
    # --- Batch Size & Accumulation ---
    per_device_train_batch_size=4,   # Basso per VRAM 8GB
    gradient_accumulation_steps=8,   # Effettivo = 4*8 = 32
    per_device_eval_batch_size=8,
    
    # --- Learning Rate & Ottimizzazione ---
    learning_rate=2e-5,              # Basso perché stiamo finetunando pochi layer [cite: 329]
    weight_decay=0.1,                # Alto per regolarizzare (default è 0.01)
    num_train_epochs=1,              # Inizia con 1 epoca. Con 200k+ sample, 1 epoca è tanta roba.
    warmup_ratio=0.1,
    
    # --- Logging & Eval ---
    logging_steps=50,                # Vedi loss spesso
    eval_strategy="steps",
    eval_steps=250,                  # Valida ogni 250 step (circa 4 volte a epoca)
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # --- Precisione ---
    bf16=bf16_supported(),           # Usa BF16 se RTX 30xx, altrimenti FP16
    fp16=not bf16_supported(),
    
    # --- Vari ---
    report_to="none",
    dataloader_num_workers=2,        # Velocizza caricamento dati
    group_by_length=True,            # Ottimizza padding
)

trainer = WeightedBCETrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    tokenizer=tokenizer,             # IMPORTANTE per salvare il tokenizer alla fine
    compute_metrics=compute_metrics  # Ora vediamo Accuracy/F1!
)

print("🚀 Starting Training...")
trainer.train()

# Salvataggio finale
final_path = paths.checkpoints / "teacher_bert_large_final"
trainer.save_model(str(final_path))
print(f"Model saved to {final_path}")

C:\Users\cola0\AppData\Local\Temp\ipykernel_29008\2090560672.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCETrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedBCETrainer(


🚀 Starting Training...


c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc
250,0.679500,0.654605,0.646289,0.099150,0.391108,0.056771,0.518538
500,0.631500,0.638735,0.656790,0.003919,0.400000,0.001969,0.579029


c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


KeyboardInterrupt: 